In [1]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, ArrayType,
    LongType, DoubleType, IntegerType, DecimalType
)
from utils.spark_session import createSpark

PRICE_T = DecimalType(28, 12)  # шиткоины с ценой 5e-05 — нужна точность

# Сырые типы — как реально приходят (числа JSON parse'ятся в double, ts в long)
ohlc_row_schema = ArrayType(DoubleType())  # [ts, o, h, l, c]
payload_schema = StructType([
    StructField("coin_id",      StringType()),
    StructField("vs_currency",  StringType()),
    StructField("days",         IntegerType()),
    StructField("fetched_at",   LongType()),
    StructField("ohlc",         ArrayType(ohlc_row_schema)),
])

TOPIC = "crypto.ohlcv"
CHECKPOINT_BRONZE_PATH = f"s3a://spark-checkpoints/bronze/{TOPIC}"
CHECKPOINT_SILVER_PATH = f"s3a://spark-checkpoints/silver/{TOPIC}"
BRONZE_PATH = f"s3a://crypto-lake/bronze/{TOPIC}"
SILVER_PATH = f"s3a://crypto-lake/silver/{TOPIC}"

def days_to_timeframe(days_col):
    return (F.when(days_col <= 1, "30m")
             .when(days_col <= 30, "4h")
             .otherwise("4d"))

spark = createSpark()

df = spark.read.parquet(BRONZE_PATH)
bronze_schema = df.schema  
print(bronze_schema)

bronze = (spark.readStream
    .schema(bronze_schema)
    .option("maxFilesPerTrigger", 50)
    .parquet(BRONZE_PATH))

clean = (bronze
    .withColumn("p", F.from_json("raw_json", payload_schema))
    .where(F.size("p.ohlc") > 0)                          # фильтр пустых ответов
    .select(
        "ingestion_ts", "kafka_ts",
        F.col("p.coin_id").alias("coin_id"),
        F.col("p.vs_currency").alias("vs_currency"),
        F.col("p.days").alias("days"),
        days_to_timeframe(F.col("p.days")).alias("timeframe"),
        F.explode("p.ohlc").alias("candle"),
    )
    .select(
        "ingestion_ts", "kafka_ts", "coin_id", "vs_currency", "timeframe",
        F.timestamp_millis(F.col("candle")[0].cast("long")).alias("open_time"),
        F.col("candle")[1].cast(PRICE_T).alias("open"),
        F.col("candle")[2].cast(PRICE_T).alias("high"),
        F.col("candle")[3].cast(PRICE_T).alias("low"),
        F.col("candle")[4].cast(PRICE_T).alias("close"),
    )
    .where(F.col("close") > 0)
    .withWatermark("open_time", "7 days")                 # 4d свечи => watermark щедрый
    .dropDuplicates(["coin_id", "timeframe", "open_time"])
    .withColumn("year",  F.year("open_time"))
    .withColumn("month", F.month("open_time"))
    .withColumn("day",   F.dayofmonth("open_time"))
)
'''
(clean.writeStream
    .format("parquet")
    .option("path", "s3a://crypto-lake/silver/ohlcv_clean/")
    .option("checkpointLocation", "s3a://crypto-lake/silver/_checkpoints/ohlcv_clean/")
    .partitionBy("coin_id", "year", "month")              # частые фильтры по монете и периоду
    .outputMode("append")
    .trigger(processingTime="60 seconds")
    .start()
    .awaitTermination())'''

write = (clean.writeStream
    .format("console")
    .option("truncate", False)
    #.trigger(processingTime="60 seconds")
    .start())


write.awaitTermination()

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-2b5caefd-4cb4-40ed-a0fd-7e47174508fd;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.5 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.5 in central
	found org.apache.kafka#kafka-clients;3.4.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.5 in central
	found org.slf4j#slf4j-api;2.0.7 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.commons#c

StructType([StructField('raw_json', StringType(), True), StructField('topic', StringType(), True), StructField('partition', IntegerType(), True), StructField('offset', LongType(), True), StructField('kafka_ts', TimestampType(), True), StructField('ingestion_ts', TimestampType(), False), StructField('year', IntegerType(), True), StructField('month', IntegerType(), True), StructField('day', IntegerType(), True), StructField('hour', IntegerType(), True)])


-------------------------------------------
Batch: 0
-------------------------------------------
+------------+--------+-------+-----------+---------+---------+----+----+---+-----+----+-----+---+
|ingestion_ts|kafka_ts|coin_id|vs_currency|timeframe|open_time|open|high|low|close|year|month|day|
+------------+--------+-------+-----------+---------+---------+----+----+---+-----+----+-----+---+
+------------+--------+-------+-----------+---------+---------+----+----+---+-----+----+-----+---+



-------------------------------------------
Batch: 1
-------------------------------------------
+------------+--------+-------+-----------+---------+---------+----+----+---+-----+----+-----+---+
|ingestion_ts|kafka_ts|coin_id|vs_currency|timeframe|open_time|open|high|low|close|year|month|day|
+------------+--------+-------+-----------+---------+---------+----+----+---+-----+----+-----+---+
+------------+--------+-------+-----------+---------+---------+----+----+---+-----+----+-----+---+



-------------------------------------------
Batch: 2
-------------------------------------------
+------------+--------+-------+-----------+---------+---------+----+----+---+-----+----+-----+---+
|ingestion_ts|kafka_ts|coin_id|vs_currency|timeframe|open_time|open|high|low|close|year|month|day|
+------------+--------+-------+-----------+---------+---------+----+----+---+-----+----+-----+---+
+------------+--------+-------+-----------+---------+---------+----+----+---+-----+----+-----+---+



-------------------------------------------
Batch: 3
-------------------------------------------
+------------+--------+-------+-----------+---------+---------+----+----+---+-----+----+-----+---+
|ingestion_ts|kafka_ts|coin_id|vs_currency|timeframe|open_time|open|high|low|close|year|month|day|
+------------+--------+-------+-----------+---------+---------+----+----+---+-----+----+-----+---+
+------------+--------+-------+-----------+---------+---------+----+----+---+-----+----+-----+---+



-------------------------------------------
Batch: 4
-------------------------------------------
+------------+--------+-------+-----------+---------+---------+----+----+---+-----+----+-----+---+
|ingestion_ts|kafka_ts|coin_id|vs_currency|timeframe|open_time|open|high|low|close|year|month|day|
+------------+--------+-------+-----------+---------+---------+----+----+---+-----+----+-----+---+
+------------+--------+-------+-----------+---------+---------+----+----+---+-----+----+-----+---+



26/05/04 14:45:55 ERROR TorrentBroadcast: Store broadcast broadcast_23 fail, remove all pieces of the broadcast
26/05/04 14:45:56 ERROR MicroBatchExecution: Query [id = 9ac29ca5-297b-47d7-a582-832391e9eaaa, runId = af58684d-67ec-45af-b400-c84a316e9bb6] terminated with error
java.lang.OutOfMemoryError: Java heap space
	at java.base/java.nio.HeapByteBuffer.<init>(Unknown Source)
	at java.base/java.nio.ByteBuffer.allocate(Unknown Source)
	at org.apache.spark.broadcast.TorrentBroadcast$.$anonfun$blockifyObject$1(TorrentBroadcast.scala:360)
	at org.apache.spark.broadcast.TorrentBroadcast$.$anonfun$blockifyObject$1$adapted(TorrentBroadcast.scala:360)
	at org.apache.spark.broadcast.TorrentBroadcast$$$Lambda$1748/0x0000000840dcb840.apply(Unknown Source)
	at org.apache.spark.util.io.ChunkedByteBufferOutputStream.allocateNewChunkIfNeeded(ChunkedByteBufferOutputStream.scala:87)
	at org.apache.spark.util.io.ChunkedByteBufferOutputStream.write(ChunkedByteBufferOutputStream.scala:75)
	at net.jpountz

StreamingQueryException: [STREAM_FAILED] Query [id = 9ac29ca5-297b-47d7-a582-832391e9eaaa, runId = af58684d-67ec-45af-b400-c84a316e9bb6] terminated with exception: Java heap space

In [5]:
# вместо readStream — read
bronze = spark.read.parquet(BRONZE_PATH)

clean = (bronze
    .withColumn("p", F.from_json("raw_json", payload_schema))
    .where(F.size("p.ohlc") > 0)
    .select(
        "ingestion_ts", "kafka_ts",
        F.col("p.coin_id").alias("coin_id"),
        F.col("p.vs_currency").alias("vs_currency"),
        F.col("p.days").alias("days"),
        days_to_timeframe(F.col("p.days")).alias("timeframe"),
        F.explode("p.ohlc").alias("candle"),
    )
    .select(
        "ingestion_ts", "kafka_ts", "coin_id", "vs_currency", "timeframe",
        F.timestamp_millis(F.col("candle")[0].cast("long")).alias("open_time"),
        F.col("candle")[1].cast(PRICE_T).alias("open"),
        F.col("candle")[2].cast(PRICE_T).alias("high"),
        F.col("candle")[3].cast(PRICE_T).alias("low"),
        F.col("candle")[4].cast(PRICE_T).alias("close"),
    )
    .where(F.col("close") > 0)
    .dropDuplicates(["coin_id", "timeframe", "open_time"])
    .withColumn("year",  F.year("open_time"))
    .withColumn("month", F.month("open_time"))
    .withColumn("day",   F.dayofmonth("open_time"))
)

clean.show(20, truncate=False)
clean.printSchema()
print("rows:", clean.count())

+------------+--------+-------+-----------+---------+---------+----+----+---+-----+----+-----+---+
|ingestion_ts|kafka_ts|coin_id|vs_currency|timeframe|open_time|open|high|low|close|year|month|day|
+------------+--------+-------+-----------+---------+---------+----+----+---+-----+----+-----+---+
+------------+--------+-------+-----------+---------+---------+----+----+---+-----+----+-----+---+

root
 |-- ingestion_ts: timestamp (nullable = false)
 |-- kafka_ts: timestamp (nullable = true)
 |-- coin_id: string (nullable = true)
 |-- vs_currency: string (nullable = true)
 |-- timeframe: string (nullable = false)
 |-- open_time: timestamp (nullable = true)
 |-- open: decimal(28,12) (nullable = true)
 |-- high: decimal(28,12) (nullable = true)
 |-- low: decimal(28,12) (nullable = true)
 |-- close: decimal(28,12) (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- day: integer (nullable = true)

rows: 0


-------------------------------------------
Batch: 8
-------------------------------------------
+------------+--------+-------+-----------+---------+---------+----+----+---+-----+----+-----+---+
|ingestion_ts|kafka_ts|coin_id|vs_currency|timeframe|open_time|open|high|low|close|year|month|day|
+------------+--------+-------+-----------+---------+---------+----+----+---+-----+----+-----+---+
+------------+--------+-------+-----------+---------+---------+----+----+---+-----+----+-----+---+



-------------------------------------------
Batch: 9
-------------------------------------------
+------------+--------+-------+-----------+---------+---------+----+----+---+-----+----+-----+---+
|ingestion_ts|kafka_ts|coin_id|vs_currency|timeframe|open_time|open|high|low|close|year|month|day|
+------------+--------+-------+-----------+---------+---------+----+----+---+-----+----+-----+---+
+------------+--------+-------+-----------+---------+---------+----+----+---+-----+----+-----+---+



-------------------------------------------
Batch: 10
-------------------------------------------
+------------+--------+-------+-----------+---------+---------+----+----+---+-----+----+-----+---+
|ingestion_ts|kafka_ts|coin_id|vs_currency|timeframe|open_time|open|high|low|close|year|month|day|
+------------+--------+-------+-----------+---------+---------+----+----+---+-----+----+-----+---+
+------------+--------+-------+-----------+---------+---------+----+----+---+-----+----+-----+---+

